## Data Preparation


In [1]:
# Transforming Coordinates

import pandas as pd

# Loading dataset
fb = pd.read_csv("Third_Generation_Simulation_Data__TGSIM__Foggy_Bottom_Trajectories.csv")

# Calculating H in meters
H_fb = 19399                           # difference between min and max y coordinate 
conv_fb = 0.0186613838586    # from documentation
H_meters_fb = H_fb * conv_fb

# Applying transformations to y-coordinates, y-speeds and y-accelerations in the FB dataset
fb["yloc_kf"] = H_meters_fb - fb["yloc_kf"]
fb["speed_kf_y"] = -fb["speed_kf_y"]
fb["acceleration_kf_y"] = -fb["acceleration_kf_y"]

# Saving transformed dataset
fb.to_csv("Transformed_TGSIM_Foggy_Bottom.csv", index=False)

In [2]:
# Smoothing and calculating scalar speed and acceleration
import pandas as pd
import numpy as np
from scipy.ndimage import gaussian_filter1d

# Loading and sorting dataset
fb = pd.read_csv("Transformed_TGSIM_Foggy_Bottom.csv")
fb = fb.groupby("id").filter(lambda g: len(g) > 1)
fb = fb.sort_values(by=["id", "time"]).reset_index(drop=True)


# Gaussian smoother
def gaussian_smooth(series, sigma=1.0):
    return pd.Series(
        gaussian_filter1d(series.to_numpy(), sigma=sigma, mode="nearest"),
        index=series.index
    )

# Step 1: smooth positions
fb["xloc_kf_smooth"] = fb.groupby("id")["xloc_kf"].transform(lambda s: gaussian_smooth(s, sigma=1.0))
fb["yloc_kf_smooth"] = fb.groupby("id")["yloc_kf"].transform(lambda s: gaussian_smooth(s, sigma=1.0))

# Step 2: derive velocity components
def compute_velocity(group):
    x = group["xloc_kf_smooth"].values
    y = group["yloc_kf_smooth"].values
    dt = 0.1
    vx, vy = np.zeros(len(group)), np.zeros(len(group))

    for i in range(len(group)):
        if i == 0:
            vx[i] = (x[i+1] - x[i]) / dt
            vy[i] = (y[i+1] - y[i]) / dt
        elif i == len(group)-1:
            vx[i] = (x[i] - x[i-1]) / dt
            vy[i] = (y[i] - y[i-1]) / dt
        else:
            vx[i] = (x[i+1] - x[i-1]) / (2*dt)
            vy[i] = (y[i+1] - y[i-1]) / (2*dt)

    group["speed_kf_x"] = gaussian_filter1d(vx, sigma=1.0, mode="nearest")
    group["speed_kf_y"] = gaussian_filter1d(vy, sigma=1.0, mode="nearest")
    return group

fb = fb.groupby("id", group_keys=False).apply(compute_velocity)

# Step 3: derive acceleration components
def compute_accel(group):
    vx = group["speed_kf_x"].values
    vy = group["speed_kf_y"].values
    dt = 0.1
    ax, ay = np.zeros(len(group)), np.zeros(len(group))

    for i in range(len(group)):
        if i == 0:
            ax[i] = (vx[i+1] - vx[i]) / dt
            ay[i] = (vy[i+1] - vy[i]) / dt
        elif i == len(group)-1:
            ax[i] = (vx[i] - vx[i-1]) / dt
            ay[i] = (vy[i] - vy[i-1]) / dt
        else:
            ax[i] = (vx[i+1] - vx[i-1]) / (2*dt)
            ay[i] = (vy[i+1] - vy[i-1]) / (2*dt)

    group["acceleration_kf_x"] = gaussian_filter1d(ax, sigma=1.0, mode="nearest")
    group["acceleration_kf_y"] = gaussian_filter1d(ay, sigma=1.0, mode="nearest")
    return group

fb = fb.groupby("id", group_keys=False).apply(compute_accel)

# Step 4: signed scalar acceleration and speed
def signed_accel(group):
    vx, vy = group["speed_kf_x"].to_numpy(), group["speed_kf_y"].to_numpy()
    ax, ay = group["acceleration_kf_x"].to_numpy(), group["acceleration_kf_y"].to_numpy()
    speed = np.hypot(vx, vy)
    signed = np.zeros(len(group))
    nz = speed > 0
    signed[nz] = (vx[nz]*ax[nz] + vy[nz]*ay[nz]) / speed[nz]
    group["acceleration_kf_scalar"] = signed

    # now speed
    group["speed_kf_smooth"] = np.sqrt(vx**2 + vy**2)
    return group

fb = fb.groupby("id", group_keys=False).apply(signed_accel)

# Save
fb.to_csv("Transformed_TGSIM_Foggy_Bottom_smoothed.csv", index=False)
print("Smoothed positions, velocity, and acceleration (x, y, signed scalar) computed.")

C:\Users\msela\AppData\Local\Temp\ipykernel_23212\1610122503.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fb = fb.groupby("id", group_keys=False).apply(compute_velocity)
C:\Users\msela\AppData\Local\Temp\ipykernel_23212\1610122503.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fb = fb.groupby("id", group_keys=False).apply(compute_accel)
C:\Users\msela\AppData\Local\Temp\ipykernel_23212\1610122

Smoothed positions, velocity, and acceleration (x, y, signed scalar) computed.


## Detecting Surrounding Agents for Behavior and Context Variable Computation

In [1]:
import pandas as pd
import numpy as np

# ======================================
# Load full smoothed dataset
# ======================================
tgsim = pd.read_csv("Transformed_TGSIM_Foggy_Bottom_smoothed.csv")

# Define window id (1000-second bins)
tgsim["window_id"] = (tgsim["time"] // 1000).astype(int)

# Tesla zones (unchanged)
tesla_zones = {
    "Wide Fwd": {"radius": 60, "angle": 120, "from": 300, "to": 60},
    "Main Fwd": {"radius": 150, "angle": 45, "from": 337.5, "to": 22.5},
    "Narrow Fwd": {"radius": 250, "angle": 35, "from": 342.5, "to": 17.5},
    "Side Fwd L": {"radius": 80, "angle": 90, "from": 25, "to": 115},
    "Side Fwd R": {"radius": 80, "angle": 90, "from": 245, "to": 335},
    "Rear": {"radius": 50, "angle": 135, "from": 112.5, "to": 247.5},
    "Side Rear L": {"radius": 100, "angle": 75, "from": 107.5, "to": 182.5},
    "Side Rear R": {"radius": 100, "angle": 75, "from": 177.5, "to": 252.5}
}

def normalize_angle(angle):
    return angle % 360

def rotate_zones(zones, heading):
    rotated = {}
    for name, params in zones.items():
        rotated[name] = {
            "radius": params["radius"],
            "from": normalize_angle(params["from"] + heading),
            "to": normalize_angle(params["to"] + heading)
        }
    return rotated

def detect_zones(rel_angle, distance, zones):
    hits = []
    rel_angle = normalize_angle(rel_angle)
    for zone, params in zones.items():
        f, t = params["from"], params["to"]
        if f > t:
            within = (rel_angle >= f or rel_angle <= t)
        else:
            within = (f <= rel_angle <= t)
        if within and distance <= params["radius"]:
            hits.append(zone)
    return hits

# Compute heading once
tgsim["heading"] = np.degrees(np.arctan2(
    tgsim["speed_kf_y"],
    tgsim["speed_kf_x"]
))
tgsim["heading"] = tgsim.groupby("id")["heading"].transform(lambda x: x.ffill().bfill())
tgsim["heading"] = tgsim["heading"].apply(normalize_angle)

# ======================================
# Process each window separately
# ======================================
for window in sorted(tgsim["window_id"].unique()):

    print(f"Processing window {window}")

    sub = tgsim[tgsim["window_id"] == window]

    hdv_ids = sub[sub["type_most_common"] == 3]["id"].unique()
    agent_data = sub[sub["type_most_common"].isin([0,1,2,3,4,5,6,7])]

    detection_details = []

    for timestep in sorted(sub["time"].unique()):

        hdv_positions = sub[
            (sub["time"] == timestep) &
            (sub["type_most_common"] == 3)
        ]

        agent_positions = agent_data[agent_data["time"] == timestep]

        for _, hdv_row in hdv_positions.iterrows():

            heading = hdv_row["heading"]
            rotated = rotate_zones(tesla_zones, heading)

            for _, agent_row in agent_positions.iterrows():

                dx = agent_row["xloc_kf_smooth"] - hdv_row["xloc_kf_smooth"]
                dy = agent_row["yloc_kf_smooth"] - hdv_row["yloc_kf_smooth"]

                distance = np.sqrt(dx**2 + dy**2)
                if distance < 0.01:
                    continue

                rel_angle = np.degrees(np.arctan2(dy, dx))
                zones_hit = detect_zones(rel_angle, distance, rotated)

                for zone in zones_hit:
                    detection_details.append({
                        "time": timestep,
                        "hdv_id": int(hdv_row["id"]),
                        "agent_id": int(agent_row["id"]),
                        "zone": zone,
                        "distance": distance
                    })

    # Save per-window detection file
    pd.DataFrame(detection_details).to_csv(
        f"HDV_Detection_Details_FB_window_{window}.csv",
        index=False
    )

    print(f"Window {window} saved.")


Processing window 0
Window 0 saved.
Processing window 1
Window 1 saved.
Processing window 2
Window 2 saved.
Processing window 3
Window 3 saved.
Processing window 4
Window 4 saved.
Processing window 5
Window 5 saved.
Processing window 6
Window 6 saved.
Processing window 7
Window 7 saved.


## Defining Leader and Adding Behavior Variables

In [3]:
import pandas as pd
import numpy as np
import glob

# ===============================
# Load smoothed dataset once
# ===============================
df = pd.read_csv("Transformed_TGSIM_Foggy_Bottom_smoothed.csv")

# ===============================
# Base HDV dataset (RENAMED columns kept)
# ===============================
hdv_base = df[df["type_most_common"] == 3][[
    "id", "time",
    "lane_kf",
    "speed_kf_smooth",
    "xloc_kf_smooth",
    "yloc_kf_smooth",
    "speed_kf_x",
    "speed_kf_y",
    "acceleration_kf_scalar",
    "acceleration_kf_x",
    "acceleration_kf_y"
]].rename(columns={
    "id": "hdv_id",
    "lane_kf": "lane_hdv",
    "speed_kf_smooth": "v_hdv",
    "xloc_kf_smooth": "xloc_hdv",
    "yloc_kf_smooth": "yloc_hdv",
    "speed_kf_x": "vx_hdv",
    "speed_kf_y": "vy_hdv",
    "acceleration_kf_scalar": "a_hdv",
    "acceleration_kf_x": "ax_hdv",
    "acceleration_kf_y": "ay_hdv"
})

# ===============================
# Add jerk (backward difference)
# ===============================
TIME_STEP = 0.1

hdv_base = hdv_base.sort_values(["hdv_id", "time"])

# Backward jerk
hdv_base["jerk_hdv"] = (
    hdv_base.groupby("hdv_id")["a_hdv"]
    .diff() / TIME_STEP
)

# Forward jerk for first timestep of each vehicle
forward_jerk = (
    hdv_base.groupby("hdv_id")["a_hdv"]
    .diff(-1) / -TIME_STEP
)

# Replace NaNs (first timestep) with forward jerk
hdv_base["jerk_hdv"] = hdv_base["jerk_hdv"].fillna(forward_jerk)

# Optional: if any remain (single-frame vehicles)
hdv_base["jerk_hdv"] = hdv_base["jerk_hdv"].fillna(0)

# ===============================
# Leader info (RENAMED)
# ===============================
leader_info = df[[
    "id", "time",
    "lane_kf",
    "speed_kf_smooth",
    "type_most_common",
    "xloc_kf_smooth",
    "yloc_kf_smooth",
    "speed_kf_x",
    "speed_kf_y",
    "acceleration_kf_scalar",
    "acceleration_kf_x",
    "acceleration_kf_y"
]].rename(columns={
    "id": "agent_id",
    "lane_kf": "lane_lead",
    "speed_kf_smooth": "v_lead",
    "type_most_common": "leader_type",
    "xloc_kf_smooth": "xloc_lead",
    "yloc_kf_smooth": "yloc_lead",
    "speed_kf_x": "vx_lead",
    "speed_kf_y": "vy_lead",
    "acceleration_kf_scalar": "a_lead",
    "acceleration_kf_x": "ax_lead",
    "acceleration_kf_y": "ay_lead"
})

# ===============================
# Lane compatibility
# ===============================
lane_map_fb = {
    1:[1,40,15],2:[2,37,29],3:[3,48,34],4:[4,42,25],
    15:[15,46,3],25:[25,36,2],29:[29,41,1],34:[34,45,4],
    36:[36,2,37],37:[37,29],40:[40,15],41:[41,1,40],
    42:[42,25],45:[45,4,42],46:[46,3,48],48:[48,34]
}

lane_pairs = pd.DataFrame(
    [(k, v) for k, vals in lane_map_fb.items() for v in vals],
    columns=["lane_hdv", "lane_lead"]
)

forward_zones = {"Wide Fwd", "Main Fwd", "Narrow Fwd", "Side Fwd L"}

# ===============================
# Process detection windows
# ===============================
leader_results = []

detection_files = sorted(glob.glob("HDV_Detection_Details_FB_window_*.csv"))

for file in detection_files:
    det = pd.read_csv(file)

    det = det[det["zone"].isin(forward_zones)]
    det = det[det["hdv_id"] != det["agent_id"]]

    det = det.merge(hdv_base, on=["hdv_id", "time"], how="left")
    
    det = det.merge(leader_info, on=["agent_id", "time"], how="left")

    # Keep only valid leader vehicle types
    valid_leader_types = {3, 4, 5, 6, 7}
    det = det[det["leader_type"].isin(valid_leader_types)]
    
    det = det.merge(lane_pairs, on=["lane_hdv", "lane_lead"], how="inner")

    det = det.sort_values("distance")
    leaders = det.groupby(["hdv_id", "time"], as_index=False).first()

    leaders["headway_s"] = leaders["distance"]
    leaders["delta_v"] = leaders["v_hdv"] - leaders["v_lead"]
    leaders["headway_t"] = np.where(
        leaders["v_hdv"] > 0,
        leaders["headway_s"] / leaders["v_hdv"],
        np.nan
    )

    leaders = leaders[[
        "hdv_id", "time",
        "leader_id" if "leader_id" in leaders.columns else "agent_id",
        "leader_type",
        "headway_s",
        "headway_t",
        "delta_v"
    ]]

    leaders = leaders.rename(columns={"agent_id": "leader_id"})

    leader_results.append(leaders)

# ===============================
# Combine windows
# ===============================
leaders_all = pd.concat(leader_results, ignore_index=True)

leaders_all = leaders_all.groupby(
    ["hdv_id", "time"], as_index=False
).first()

# ===============================
# Final dataset (renamed columns preserved)
# ===============================
final_df = hdv_base.merge(
    leaders_all,
    on=["hdv_id", "time"],
    how="left"
)

final_df.to_csv("TGSIM_FB_DM_Dataset.csv", index=False)

print("Done.")
print(final_df.columns)
print("Rows:", len(final_df))


Done.
Index(['hdv_id', 'time', 'lane_hdv', 'v_hdv', 'xloc_hdv', 'yloc_hdv', 'vx_hdv',
       'vy_hdv', 'a_hdv', 'ax_hdv', 'ay_hdv', 'jerk_hdv', 'leader_id',
       'leader_type', 'headway_s', 'headway_t', 'delta_v'],
      dtype='object')
Rows: 1581551


## Adding Context Variables

### Distance to nearest pedestrian in the forward zones

In [4]:
import pandas as pd

# 1. Load main dataset
dm = pd.read_csv("TGSIM_FB_DM_Dataset.csv")

# 2. Load detection windows
detections = pd.concat(
    [pd.read_csv(f"HDV_Detection_Details_FB_window_{i}.csv")
     for i in range(8)],
    ignore_index=True
)

# 3. Enforce exact 0.1s alignment (eliminate float precision issues)
dm["time"] = (dm["time"] * 10).round().astype(int) / 10
detections["time"] = (detections["time"] * 10).round().astype(int) / 10

# 4. Keep forward zones
forward_zones = [
    "Wide Fwd", "Main Fwd", "Narrow Fwd",
    "Side Fwd L", "Side Fwd R"
]
detections = detections[detections["zone"].isin(forward_zones)]

# 5. Keep pedestrians only
full_data = pd.read_csv("Transformed_TGSIM_Foggy_Bottom_smoothed.csv")
ped_ids = set(full_data.loc[
    full_data["type_most_common"] == 0, "id"
])
detections = detections[detections["agent_id"].isin(ped_ids)]

# 6. Remove duplicate zone overlaps (same agent same timestep)
detections = detections.drop_duplicates(
    subset=["hdv_id", "time", "agent_id"]
)

# 7. Compute nearest pedestrian distance
d_ped_df = (
    detections
    .groupby(["hdv_id", "time"], as_index=False)["distance"]
    .min()
    .rename(columns={"distance": "d_ped"})
)

# 8. Final safety: enforce unique merge keys
d_ped_df = d_ped_df.drop_duplicates(["hdv_id", "time"])

# 9. Merge back (row count will remain identical)
dm = dm.merge(
    d_ped_df,
    on=["hdv_id", "time"],
    how="left"
)

# 10. Save
dm.to_csv("TGSIM_FB_DM_Dataset_with_dped.csv", index=False)

print("Done.")


Done.


### Distance to stop line

In [5]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load main dataset (already contains d_ped)
# --------------------------------------------------
dm = pd.read_csv("TGSIM_FB_DM_Dataset_with_dped.csv")

# Enforce exact 0.1 s alignment again (safety)
dm["time"] = (dm["time"] * 10).round().astype(int) / 10


# --------------------------------------------------
# 2. Load stop line coordinates
# --------------------------------------------------
stop_lines = pd.read_csv("Stop_Lines.csv")

# Ensure expected column names
# Expected: lane_kf, x, y
# Rename if needed
stop_lines = stop_lines.rename(columns={
    "x": "stop_x",
    "y": "stop_y"
})

# Drop duplicates if any
stop_lines = stop_lines.drop_duplicates(subset=["lane_kf"])


# --------------------------------------------------
# 3. Merge stop coordinates by lane
# --------------------------------------------------
dm = dm.merge(
    stop_lines[["lane_kf", "stop_x", "stop_y"]],
    left_on="lane_hdv",
    right_on="lane_kf",
    how="left"
)

# --------------------------------------------------
# 4. Compute Euclidean distance to stop line
# --------------------------------------------------
# Only compute where stop coordinates exist
mask = dm["stop_x"].notna() & dm["stop_y"].notna()

dm["d_stop"] = np.nan  # default

dm.loc[mask, "d_stop"] = np.sqrt(
    (dm.loc[mask, "xloc_hdv"] - dm.loc[mask, "stop_x"])**2 +
    (dm.loc[mask, "yloc_hdv"] - dm.loc[mask, "stop_y"])**2
)

# --------------------------------------------------
# 5. Cleanup
# --------------------------------------------------
dm = dm.drop(columns=["lane_kf", "stop_x", "stop_y"], errors="ignore")

# --------------------------------------------------
# 6. Save updated dataset
# --------------------------------------------------
dm.to_csv("TGSIM_FB_DM_Dataset_with_dped_dstop.csv", index=False)

print("d_stop successfully added.")


d_stop successfully added.


### Local density and average speed

In [6]:
import pandas as pd

# --------------------------------------------------
# 1. Load main dataset
# --------------------------------------------------
dm = pd.read_csv("TGSIM_FB_DM_Dataset_with_dped_dstop.csv")
dm["time"] = (dm["time"] * 10).round().astype(int) / 10

# --------------------------------------------------
# 2. Load trajectory dataset once
# --------------------------------------------------
traj = pd.read_csv("Transformed_TGSIM_Foggy_Bottom_smoothed.csv")
traj["time"] = (traj["time"] * 10).round().astype(int) / 10

motorized_types = {3, 4, 5, 6, 7}
traj_motorized = traj[traj["type_most_common"].isin(motorized_types)][
    ["id", "time", "speed_kf_smooth"]
].rename(columns={
    "id": "agent_id",
    "speed_kf_smooth": "agent_speed"
})

eligible_zones = {
    "Wide Fwd",
    "Main Fwd",
    "Narrow Fwd",
    "Side Fwd L",
    "Side Fwd R"
}

all_density = []
all_avg_v = []

# --------------------------------------------------
# 3. Single pass through windows
# --------------------------------------------------
for window_id in range(8):

    print(f"Processing window {window_id}...")

    detections = pd.read_csv(
        f"HDV_Detection_Details_FB_window_{window_id}.csv"
    )

    detections["time"] = (
        detections["time"] * 10
    ).round().astype(int) / 10

    # Remove duplicate overlaps
    detections = detections.drop_duplicates(
        subset=["hdv_id", "time", "agent_id"]
    )

    # -------- Density --------
    density_df = (
        detections
        .groupby(["hdv_id", "time"])["agent_id"]
        .nunique()
        .reset_index()
        .rename(columns={"agent_id": "density"})
    )
    all_density.append(density_df)

    # -------- avg_v --------
    det_forward = detections[detections["zone"].isin(eligible_zones)]

    det_forward = det_forward.merge(
        traj_motorized,
        on=["agent_id", "time"],
        how="inner"
    )

    avg_v_df = (
        det_forward
        .groupby(["hdv_id", "time"])["agent_speed"]
        .mean()
        .reset_index()
        .rename(columns={"agent_speed": "avg_v"})
    )

    all_avg_v.append(avg_v_df)

print("All windows processed.")

# --------------------------------------------------
# 4. Combine results
# --------------------------------------------------
density_full = (
    pd.concat(all_density, ignore_index=True)
    .drop_duplicates(["hdv_id", "time"])
)

avg_v_full = (
    pd.concat(all_avg_v, ignore_index=True)
    .drop_duplicates(["hdv_id", "time"])
)

# --------------------------------------------------
# 5. Merge once
# --------------------------------------------------
dm = dm.merge(density_full, on=["hdv_id", "time"], how="left")
dm = dm.merge(avg_v_full, on=["hdv_id", "time"], how="left")

dm["density"] = dm["density"].fillna(0).astype(int)
dm["avg_v"] = dm["avg_v"].fillna(0)

# --------------------------------------------------
# 6. Save
# --------------------------------------------------
dm.to_csv("TGSIM_FB_DM_Final_Dataset.csv", index=False)

print("Done.")

Processing window 0...
Processing window 1...
Processing window 2...
Processing window 3...
Processing window 4...
Processing window 5...
Processing window 6...
Processing window 7...
All windows processed.
Done.


## Handling NaNs

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("TGSIM_FB_DM_Final_Dataset.csv")

# Replace inf first
df = df.replace([np.inf, -np.inf], np.nan)

# Behavioral reference max
max_headway = df["headway_s"].quantile(0.99)

# Context reference max
max_dped = df["d_ped"].quantile(0.99)
max_dstop = df["d_stop"].quantile(0.99)

print("Reference values:")
print("max_headway:", max_headway)
print("max_dped:", max_dped)
print("max_dstop:", max_dstop)

# --------------------------------------------------
# BEHAVIORAL VARIABLES
# --------------------------------------------------

# No leader → delta_v = 0
df["delta_v"] = df["delta_v"].fillna(0)

# No leader → headway = large
df["headway_s"] = df["headway_s"].fillna(max_headway)

# --------------------------------------------------
# CONTEXT VARIABLES
# --------------------------------------------------

# No pedestrian detected → far away
df["d_ped"] = df["d_ped"].fillna(max_dped)

# No stop line → far away
df["d_stop"] = df["d_stop"].fillna(max_dstop)

print("Remaining NaNs:")
print(df[["delta_v", "headway_s", "a_hdv",
          "d_ped", "d_stop", "density"]].isna().sum())
df.to_csv("TGSIM_FB_DM_Final_Dataset.csv", index=False)

Reference values:
max_headway: 120.70353703354706
max_dped: 184.63684534087847
max_dstop: 125.4182702397102
Remaining NaNs:
delta_v      0
headway_s    0
a_hdv        0
d_ped        0
d_stop       0
density      0
dtype: int64


In [10]:
df.columns

Index(['hdv_id', 'time', 'lane_hdv', 'v_hdv', 'xloc_hdv', 'yloc_hdv', 'vx_hdv',
       'vy_hdv', 'a_hdv', 'ax_hdv', 'ay_hdv', 'jerk_hdv', 'leader_id',
       'leader_type', 'headway_s', 'headway_t', 'delta_v', 'd_ped', 'd_stop',
       'density', 'avg_v'],
      dtype='object')